# Deep Hebbian hiérarchique (L1 → L2)

Refonte de la classification : on remplace l'encodeur unique par une **hiérarchie** où L1 apprend des détecteurs de bords (petits patches) et L2 combine ces bords en formes (boucles, intersections).

## Les mécanismes
1. **Hiérarchie Deep Hebbian** : L1 (patches 4×4 → 32 bords) puis L2 (→ 64 formes).
2. **Anti-Hebbian** : deux neurones qui gagnent ensemble voient leurs poids communs réduits (décorrélation / spécialisation).
3. **Soft-WTA** (SoftHebb) : sélection probabiliste des gagnants pendant l'apprentissage, déterministe à l'inférence.

> **Note** : le saliency-gate a été retiré — il bloquait trop l'apprentissage (62% des mises à jour à eta=0). On apprend sur chaque entrée (Deep Hebbian pur).

## 0. Imports

In [1]:
# Deep Hebbian hiérarchique — L1 (bords) → L2 (formes)
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, train_readout, DeepHebbian,
    anti_hebbian_update, saliency_gate, soft_wta, AnchorNeurons)

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Encodeur Deep Hebbian

In [3]:
enc = DeepHebbian(patch_l1=7, n_l1=64, n_l2=128, lr_l1=0.1, lr_l2=0.1, seed=0,
                  temp=0.5, n_learn=8, lr_anti=0.01)

# Entraînement non supervisé (surprise variable)
print("=== Entraînement Deep Hebbian (non supervisé) ===")
cnt = [0]*10
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if cnt[l] >= 30: continue
    S = 0.3 + 0.5*np.random.rand()
    enc.encode(train_set[i][0].squeeze().numpy(), learn=True, S=S)
    cnt[l] += 1
    if sum(cnt) >= 300: break
print("  entraîné sur 300 images (L1 + L2)")

# Vérifier que les deux couches ont appris
print(f"  spécialisation W1 (std lignes) : {np.std(enc.W1, axis=0).mean():.3f}")
print(f"  spécialisation W2 (std lignes) : {np.std(enc.W2, axis=0).mean():.3f}")

=== Entraînement Deep Hebbian (non supervisé) ===


  entraîné sur 300 images (L1 + L2)
  spécialisation W1 (std lignes) : 0.140
  spécialisation W2 (std lignes) : 0.117


## 3. Classification (couche lue supervisée + ancres non supervisées)

In [4]:
def extract(dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(enc.encode(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

Xtr, ytr = extract(train_set, 400)
Xte, yte = extract(test_set, 200)
print(f"Signature hiérarchique : {Xtr.shape}")

# supervisé (couche lue)
ro = train_readout(Xtr, ytr, n_classes=10, epochs=100)
with torch.no_grad():
    acc_sup = (ro(torch.tensor(Xte,dtype=torch.float32)).argmax(1)==torch.tensor(yte)).float().mean().item()
print(f"\nClassification supervisée (couche lue) : {acc_sup:.3f}")

# non supervisé (ancres)
anchors = AnchorNeurons(d_in=Xtr.shape[1], n_neurons=50, seed=0, lr=0.1, use_homeostasis=True)
for i in range(len(Xtr)):
    anchors.learn(Xtr[i], k=3, label=ytr[i])
correct = total = 0
for i in range(len(Xte)):
    p, _ = anchors.predict_label(Xte[i])
    if p is not None:
        total += 1
        if p == yte[i]: correct += 1
acc_unsup = correct/total if total else 0
print(f"Classification non supervisée (ancres) : {acc_unsup:.3f}")

Signature hiérarchique : (400, 192)



Classification supervisée (couche lue) : 0.395
Classification non supervisée (ancres) : 0.170


## 4. Analyse honnête

In [5]:
print("=== ANALYSE HONNÊTE : DEEP HEBBIAN ===")
print(f"  Supervisé (couche lue) : {acc_sup:.3f}")
print(f"  Non supervisé (ancres) : {acc_unsup:.3f}")
print()
print("Le Deep Hebbian hiérarchique (L1→L2) ne dépasse PAS l'encodeur simple")
print("(0.747). Le plafond est ~0.37 même avec un encodeur plus large.")
print()
print("CAUSES PROBABLES :")
print("1. Le soft-WTA probabiliste + l'agrégation L1→L2 PAR SOMME détruisent la")
print("   STRUCTURE SPATIALE (la position des bords est perdue).")
print("2. La littérature (HMAX, PCANet) garde des FEATURE MAPS positionnées")
print("   entre les couches — pas une somme globale.")
print("3. L'anti-Hebbian et la sparsité réduisent encore la capacité.")
print()
print("=> La hiérarchie est le bon concept mais nécessite de préserver l'espace")
print("   (feature maps) entre L1 et L2, pas une agrégation par somme.")

=== ANALYSE HONNÊTE : DEEP HEBBIAN ===
  Supervisé (couche lue) : 0.395
  Non supervisé (ancres) : 0.170

Le Deep Hebbian hiérarchique (L1→L2) ne dépasse PAS l'encodeur simple
(0.747). Le plafond est ~0.37 même avec un encodeur plus large.

CAUSES PROBABLES :
1. Le soft-WTA probabiliste + l'agrégation L1→L2 PAR SOMME détruisent la
   STRUCTURE SPATIALE (la position des bords est perdue).
2. La littérature (HMAX, PCANet) garde des FEATURE MAPS positionnées
   entre les couches — pas une somme globale.
3. L'anti-Hebbian et la sparsité réduisent encore la capacité.

=> La hiérarchie est le bon concept mais nécessite de préserver l'espace
   (feature maps) entre L1 et L2, pas une agrégation par somme.
